# Exploratory Data Analysis — Fashion Product Images (small)

Phase 1 EDA for the multi-modal classifier. We inspect the `subCategory`
distribution, confirm the **top-10** target classes, check for missing
images / malformed rows, and preview a few `(image, text, label)` triples.

## 0. Setup — clone the repo & install deps

Runs on **Google Colab**: set the runtime to **GPU** first
(`Runtime → Change runtime type → T4 GPU`). This cell is safe to re-run.

In [ ]:
import os
if not os.path.isdir('/content/DeepLearningProject'):
    !git clone -b claude/build-multimodal-classifier https://github.com/yigitdagidir/DeepLearningProject.git /content/DeepLearningProject
%cd /content/DeepLearningProject
!git pull --ff-only   # pick up any new commits on re-run
!pip -q install -r requirements.txt

## 0b. Kaggle credentials (needed to download the dataset)

The Kaggle *Fashion Product Images (small)* dataset needs a (free) Kaggle
account + API token: **kaggle.com → your avatar → Settings → API →
Create New API Token** (downloads `kaggle.json`). Run the cell below and
upload that file. *(Alternative: uncomment Option B and paste your creds.)*

In [ ]:
# Option A — upload kaggle.json:
from google.colab import files
files.upload()  # choose your kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B — set credentials directly (instead of Option A):
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src import config
config.set_seeds()
print('config OK — top-N =', config.N_CLASSES, '| seed =', config.SEED)

## 1. Download the dataset
Idempotent: re-running reuses the kagglehub cache.

In [ ]:
from src.data.download import download, resolve_dataset_paths
styles_csv, images_dir = download()
print('styles.csv:', styles_csv)
print('images dir:', images_dir)

## 2. Load `styles.csv` and inspect

In [ ]:
df = pd.read_csv(styles_csv, on_bad_lines='skip')
print('rows:', len(df))
df.head()

In [ ]:
print(df.dtypes)
df[[config.ID_COLUMN, config.LABEL_COLUMN, config.TEXT_COLUMN]].isna().sum()

## 3. `subCategory` distribution and the top-10

In [ ]:
counts = df[config.LABEL_COLUMN].value_counts()
print('total subCategory classes:', len(counts))
top = counts.head(config.N_CLASSES)
print('\nTop', config.N_CLASSES, 'classes:')
print(top)

In [ ]:
ax = top[::-1].plot(kind='barh', figsize=(8,5))
ax.set_title(f'Top-{config.N_CLASSES} subCategory classes by frequency')
ax.set_xlabel('count')
plt.tight_layout(); plt.show()

**Why top-10 `subCategory`?** `masterCategory` is too easy (fusion adds nothing
visible); `articleType` has 140+ classes (too hard). Top-10 `subCategory` is the
sweet spot where the multi-modal contribution is measurable.

## 4. Missing images / malformed rows

In [ ]:
from pathlib import Path
sample = df.head(2000)
missing = sum(not (images_dir / f"{int(i)}.jpg").is_file() for i in sample[config.ID_COLUMN])
print(f'missing images in 2k sample: {missing}')
print('blank productDisplayName:', (df[config.TEXT_COLUMN].astype(str).str.strip()=="").sum())

## 5. Preview `(image, text, label)` triples

In [ ]:
from PIL import Image
top_classes = counts.head(config.N_CLASSES).index.tolist()
view = df[df[config.LABEL_COLUMN].isin(top_classes)].sample(6, random_state=config.SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, r) in zip(axes.ravel(), view.iterrows()):
    p = images_dir / f"{int(r[config.ID_COLUMN])}.jpg"
    if p.is_file():
        ax.imshow(Image.open(p))
    ax.set_title(f"{r[config.LABEL_COLUMN]}\n{str(r[config.TEXT_COLUMN])[:30]}", fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Build the stratified splits
Writes `data/splits/{train,val,test}.csv` and persists the class list.

In [ ]:
from src.data.preprocess import preprocess
preprocess()

In [ ]:
for name, path in [('train', config.TRAIN_MANIFEST), ('val', config.VAL_MANIFEST), ('test', config.TEST_MANIFEST)]:
    s = pd.read_csv(path)
    print(f'{name:5s} n={len(s):6d}')
print('class list:', config.load_class_list())

## 7. Sanity-check the `tf.data` pipeline

In [ ]:
from src.data.dataset import make_dataset, get_text_vectorizer
vec = get_text_vectorizer()
train_ds = make_dataset(config.TRAIN_MANIFEST, 'fusion', training=True)
(img, txt), y = next(iter(train_ds))
print('image batch:', img.shape, img.dtype)
print('text  batch:', txt.shape, txt.dtype)
print('label batch:', y.shape, y.dtype)
print('example text:', txt[0].numpy().decode())
print('vectorised  :', vec(txt[:1]).numpy()[0])